# 02 Prepare Petrobras 3W Data

This notebook rebuilds a processed SCADA-style CSV from the raw Petrobras 3W parquet files and writes a processing summary JSON alongside it.

In [ ]:
import os
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

def find_repo_root() -> Path:
    env_repo = os.getenv('PIPELINE_NOTEBOOK_REPO_ROOT')
    if env_repo:
        return Path(env_repo).expanduser().resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'scripts' / 'download_petrobras_3w.py').exists():
            return candidate
    return Path('/content/workspace/pipeline-leak-detection')

REPO_ROOT = find_repo_root()
default_storage = Path('/content/drive/MyDrive/pipeline-leak-detection') if Path('/content').exists() else REPO_ROOT / 'artifacts' / 'notebook_runs'
STORAGE_ROOT = Path(os.getenv('PIPELINE_NOTEBOOK_STORAGE_ROOT', str(default_storage))).expanduser().resolve()
RUN_LABEL = os.getenv('PIPELINE_NOTEBOOK_RUN_LABEL', 'petrobras_full_run_01')
max_files_env = os.getenv('PIPELINE_NOTEBOOK_MAX_FILES')
MAX_FILES = int(max_files_env) if max_files_env not in (None, '', 'none', 'None') else 500
DOWNSAMPLE = int(os.getenv('PIPELINE_NOTEBOOK_DOWNSAMPLE', '10'))

repo_raw_root = REPO_ROOT / 'data' / 'raw' / 'petrobras_3w'
default_raw_root = STORAGE_ROOT / 'raw' / 'petrobras_3w'
RAW_ROOT = Path(os.getenv('PIPELINE_NOTEBOOK_RAW_ROOT', str(repo_raw_root if repo_raw_root.exists() else default_raw_root))).expanduser().resolve()
RUN_ROOT = STORAGE_ROOT / 'artifacts' / 'petrobras' / RUN_LABEL
PROCESSED_CSV = RUN_ROOT / 'data' / 'petrobras_3w_scada.csv'
PROCESSING_SUMMARY = RUN_ROOT / 'data' / 'petrobras_3w_processing_summary.json'

PROCESSED_CSV.parent.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'download_petrobras_3w.py'),
    '--skip-download',
    '--raw-dir',
    str(RAW_ROOT),
    '--output-csv',
    str(PROCESSED_CSV),
    '--summary-json',
    str(PROCESSING_SUMMARY),
    '--downsample',
    str(DOWNSAMPLE),
]
if MAX_FILES is not None:
    command.extend(['--max-files', str(MAX_FILES)])

subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
with open(PROCESSING_SUMMARY, 'r', encoding='utf-8') as f:
    processing_summary = json.load(f)

processing_summary


In [ ]:
pd.DataFrame(
    sorted(processing_summary['event_distribution'].items()),
    columns=['event_type', 'rows'],
).sort_values('rows', ascending=False)
